In [1]:
import os
import cv2
import shutil
import random
from tqdm import tqdm

# ==============================================================================
# 1. PENGATURAN PATH DIRECTORY
# ==============================================================================
INPUT_IMAGE_DIR = '/kaggle/input/datasets/klemenko/kitti-dataset/data_object_image_2/training/image_2' 
INPUT_LABEL_DIR = '/kaggle/input/datasets/klemenko/kitti-dataset/data_object_label_2/training/label_2'

OUTPUT_DIR = '/kaggle/working/kitti_yolo_2class'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Membuat 3 jenis split: train, val, dan test
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(OUTPUT_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, 'labels', split), exist_ok=True)

# ==============================================================================
# 2. PEMETAAN KELAS (0: Person, 1: Vehicle)
# ==============================================================================
CLASS_MAP = {
    'Pedestrian': 0, 'Person_sitting': 0, 'Cyclist': 0,
    'Car': 1, 'Van': 1, 'Truck': 1, 'Tram': 1           
}

# ==============================================================================
# 3. FUNGSI KONVERSI KITTI KE YOLO
# ==============================================================================
def convert_kitti_to_yolo(kitti_label_path, img_width, img_height):
    yolo_labels = []
    with open(kitti_label_path, 'r') as f:
        lines = f.readlines()
        
    for line in lines:
        parts = line.strip().split(' ')
        obj_class = parts[0]
        
        if obj_class not in CLASS_MAP:
            continue
            
        class_id = CLASS_MAP[obj_class]
        
        xmin = float(parts[4])
        ymin = float(parts[5])
        xmax = float(parts[6])
        ymax = float(parts[7])
        
        w = xmax - xmin
        h = ymax - ymin
        x_center = xmin + (w / 2)
        y_center = ymin + (h / 2)
        
        x_center /= img_width
        y_center /= img_height
        w /= img_width
        h /= img_height
        
        x_center = max(0.0, min(1.0, x_center))
        y_center = max(0.0, min(1.0, y_center))
        w = max(0.0, min(1.0, w))
        h = max(0.0, min(1.0, h))
        
        yolo_labels.append(f"{class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")
        
    return yolo_labels

# ==============================================================================
# 4. PROSES EKSEKUSI & SPLIT (TEST: 1000, Sisa -> 80% TRAIN, 20% VAL)
# ==============================================================================
image_files = [f for f in os.listdir(INPUT_IMAGE_DIR) if f.endswith('.png')]
random.seed(42)
random.shuffle(image_files)

# Ambil 1000 gambar untuk TEST
test_files = image_files[:1000]

# Sisanya untuk Train & Val
remaining_files = image_files[1000:]
split_idx = int(len(remaining_files) * 0.8)

train_files = remaining_files[:split_idx]
val_files = remaining_files[split_idx:]

print(f"Mempersiapkan Dataset KITTI...")
print(f"Total Gambar : {len(image_files)}")
print(f"Data Train   : {len(train_files)}")
print(f"Data Val     : {len(val_files)}")
print(f"Data Test    : {len(test_files)}")

def process_dataset(file_list, split_name):
    for filename in tqdm(file_list, desc=f"Memproses {split_name}"):
        img_path = os.path.join(INPUT_IMAGE_DIR, filename)
        label_filename = filename.replace('.png', '.txt')
        label_path = os.path.join(INPUT_LABEL_DIR, label_filename)
        
        if not os.path.exists(label_path):
            continue
            
        img = cv2.imread(img_path)
        if img is None: continue
        img_height, img_width = img.shape[:2]
        
        yolo_labels = convert_kitti_to_yolo(label_path, img_width, img_height)
        
        if len(yolo_labels) > 0:
            dst_img_path = os.path.join(OUTPUT_DIR, 'images', split_name, filename)
            shutil.copy(img_path, dst_img_path)
            
            dst_label_path = os.path.join(OUTPUT_DIR, 'labels', split_name, label_filename)
            with open(dst_label_path, 'w') as f:
                f.write('\n'.join(yolo_labels))

# Eksekusi pemrosesan file
process_dataset(train_files, 'train')
process_dataset(val_files, 'val')
process_dataset(test_files, 'test')

# ==============================================================================
# 5. BUAT FILE YAML UNTUK YOLOv8 (Ditambahkan Test Path)
# ==============================================================================
yaml_content = f"""
path: {OUTPUT_DIR}
train: images/train
val: images/val
test: images/test

names:
  0: Person
  1: Vehicle
"""

yaml_path = os.path.join(OUTPUT_DIR, 'kitti_2class.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"\n✅ Selesai! Dataset KITTI siap digunakan dengan pembagian Train/Val/Test.")
print(f"File konfigurasi YAML berada di: {yaml_path}")

Mempersiapkan Dataset KITTI...
Total Gambar : 7481
Data Train   : 5184
Data Val     : 1297
Data Test    : 1000


Memproses test: 100%|██████████| 1000/1000 [00:32<00:00, 30.86it/s]


✅ Selesai! Dataset KITTI siap digunakan dengan pembagian Train/Val/Test.
File konfigurasi YAML berada di: /kaggle/working/kitti_yolo_2class/kitti_2class.yaml


In [2]:
import os
import glob
from tqdm import tqdm

def analyze_visdrone_difficulty(label_folder, imgsz=640):
    """
    Menganalisis tingkat kesulitan dataset berdasarkan ukuran objek.
    Menggunakan definisi COCO:
    - Small:  Area < 32x32 (1024 piksel)
    - Medium: Area 32x32 s.d 96x96
    - Large:  Area > 96x96
    """

    txt_files = glob.glob(os.path.join(label_folder, "*.txt"))

    if len(txt_files) == 0:
        print("❌ Error: Tidak ada file .txt ditemukan. Cek path folder labels!")
        return

    total_images = len(txt_files)
    easy_images = 0  # Hanya Medium/Large atau Kosong
    hard_images = 0  # Ada minimal 1 Small Object
    empty_images = 0 # Tidak ada objek sama sekali

    # Ambang batas area (dalam piksel kuadrat, asumsi resize 640x640)
    # 32*32 = 1024 piksel
    SMALL_THRESHOLD = 32 * 32

    print(f"--- Menganalisis {total_images} file label ---")
    print(f"Asumsi resolusi input model: {imgsz}x{imgsz}")

    for txt_file in tqdm(txt_files):
        with open(txt_file, 'r') as f:
            lines = f.readlines()

        if len(lines) == 0:
            empty_images += 1
            easy_images += 1
            continue

        has_small_object = False

        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5: continue

            # Format YOLO: class x_center y_center width height
            # Kita butuh width dan height (normalized 0-1)
            w = float(parts[3])
            h = float(parts[4])

            # Konversi ke piksel real (berdasarkan input model 640x640)
            w_px = w * imgsz
            h_px = h * imgsz
            area_px = w_px * h_px

            if area_px < SMALL_THRESHOLD:
                has_small_object = True
                break # Cukup 1 objek kecil untuk membuat gambar jadi 'Hard'

        if has_small_object:
            hard_images += 1
        else:
            easy_images += 1

    # --- TAMPILKAN HASIL ---
    easy_pct = (easy_images / total_images) * 100
    hard_pct = (hard_images / total_images) * 100

    print("\n=== HASIL AUDIT DATASET VISDRONE (VALIDASI) ===")
    print(f"Total Gambar: {total_images}")
    print("-" * 40)
    print(f"🛑 HARD Images (Wajib P2 ON):  {hard_images} ({hard_pct:.2f}%)")
    print(f"   (Mengandung minimal 1 objek kecil < 32x32 piksel)")
    print("-" * 40)
    print(f"✅ EASY Images (Target P2 OFF): {easy_images} ({easy_pct:.2f}%)")
    print(f"   (Hanya objek sedang/besar atau kosong)")
    print(f"   *Termasuk {empty_images} gambar kosong.")
    print("-" * 40)

    print("\nKESIMPULAN UNTUK TESIS:")
    if easy_pct < 10:
        print("⚠️ Dataset ini SANGAT SULIT. Hampir semua gambar butuh P2.")
        print("   Target penghematan Anda mungkin maksimal hanya 5-10%.")
    elif easy_pct > 30:
        print("✅ Potensi efisiensi BESAR.")
        print(f"   Router Anda SEHARUSNYA bisa mematikan P2 di sekitar {easy_pct:.0f}% gambar.")
        print("   Jika aktivasi saat ini 100%, berarti tuning 'Loss Penalty' wajib dilakukan.")
    else:
        print("ℹ️ Dataset moderat. Ada potensi penghematan yang lumayan.")

In [3]:
label_path = "/kaggle/working/kitti_yolo_2class/labels/train"
analyze_visdrone_difficulty(label_path)

label_path1 = "/kaggle/working/kitti_yolo_2class/labels/val"
analyze_visdrone_difficulty(label_path1)

label_path2 = "/kaggle/working/kitti_yolo_2class/labels/test"
analyze_visdrone_difficulty(label_path2)

--- Menganalisis 5184 file label ---
Asumsi resolusi input model: 640x640


100%|██████████| 5184/5184 [00:00<00:00, 33423.32it/s]



=== HASIL AUDIT DATASET VISDRONE (VALIDASI) ===
Total Gambar: 5184
----------------------------------------
🛑 HARD Images (Wajib P2 ON):  3138 (60.53%)
   (Mengandung minimal 1 objek kecil < 32x32 piksel)
----------------------------------------
✅ EASY Images (Target P2 OFF): 2046 (39.47%)
   (Hanya objek sedang/besar atau kosong)
   *Termasuk 0 gambar kosong.
----------------------------------------

KESIMPULAN UNTUK TESIS:
✅ Potensi efisiensi BESAR.
   Router Anda SEHARUSNYA bisa mematikan P2 di sekitar 39% gambar.
   Jika aktivasi saat ini 100%, berarti tuning 'Loss Penalty' wajib dilakukan.
--- Menganalisis 1297 file label ---
Asumsi resolusi input model: 640x640


100%|██████████| 1297/1297 [00:00<00:00, 32665.29it/s]



=== HASIL AUDIT DATASET VISDRONE (VALIDASI) ===
Total Gambar: 1297
----------------------------------------
🛑 HARD Images (Wajib P2 ON):  823 (63.45%)
   (Mengandung minimal 1 objek kecil < 32x32 piksel)
----------------------------------------
✅ EASY Images (Target P2 OFF): 474 (36.55%)
   (Hanya objek sedang/besar atau kosong)
   *Termasuk 0 gambar kosong.
----------------------------------------

KESIMPULAN UNTUK TESIS:
✅ Potensi efisiensi BESAR.
   Router Anda SEHARUSNYA bisa mematikan P2 di sekitar 37% gambar.
   Jika aktivasi saat ini 100%, berarti tuning 'Loss Penalty' wajib dilakukan.
--- Menganalisis 1000 file label ---
Asumsi resolusi input model: 640x640


100%|██████████| 1000/1000 [00:00<00:00, 31005.76it/s]


=== HASIL AUDIT DATASET VISDRONE (VALIDASI) ===
Total Gambar: 1000
----------------------------------------
🛑 HARD Images (Wajib P2 ON):  620 (62.00%)
   (Mengandung minimal 1 objek kecil < 32x32 piksel)
----------------------------------------
✅ EASY Images (Target P2 OFF): 380 (38.00%)
   (Hanya objek sedang/besar atau kosong)
   *Termasuk 0 gambar kosong.
----------------------------------------

KESIMPULAN UNTUK TESIS:
✅ Potensi efisiensi BESAR.
   Router Anda SEHARUSNYA bisa mematikan P2 di sekitar 38% gambar.
   Jika aktivasi saat ini 100%, berarti tuning 'Loss Penalty' wajib dilakukan.


In [4]:
import shutil

# Ganti 'folder_name' dengan nama folder yang ingin diunduh
# Ganti 'output_file' dengan nama file zip yang diinginkan
shutil.make_archive('kitti_yolo_2class', 'zip', '/kaggle/working/kitti_yolo_2class')


'/kaggle/working/kitti_yolo_2class.zip'